# Hybrid Recommendation System Implementation

This notebook implements a hybrid recommendation system that combines collaborative filtering and content-based filtering approaches. The system will be evaluated using standard metrics and includes proper testing functionality.

## Overview
1. Data Loading and Preprocessing
2. Content-Based Filtering Implementation
3. Collaborative Filtering Implementation
4. Hybrid System Implementation
5. Evaluation and Testing

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import joblib
import os
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

## Data Loading and Preprocessing

For this example, we'll create a synthetic dataset of movies with some basic features. This could be replaced with real data like MovieLens dataset in production.

In [ ]:
# Create synthetic movie data
np.random.seed(42)

# Create movies dataframe
n_movies = 100
movie_data = {
    'movie_id': range(n_movies),
    'title': [f'Movie {i}' for i in range(n_movies)],
    'genre': np.random.choice(['Action', 'Comedy', 'Drama', 'Sci-Fi'], n_movies),
    'description': [f'This is a {np.random.choice(["exciting", "funny", "touching", "thrilling"])} movie about {np.random.choice(["adventure", "life", "space", "relationships"])}' for _ in range(n_movies)]
}
movies_df = pd.DataFrame(movie_data)

# Create users
n_users = 50
users_df = pd.DataFrame({
    'user_id': range(n_users),
    'age': np.random.randint(18, 70, n_users),
    'gender': np.random.choice(['M', 'F'], n_users)
})

# Create ratings
n_ratings = 1000
ratings_data = {
    'user_id': np.random.choice(users_df.user_id, n_ratings),
    'movie_id': np.random.choice(movies_df.movie_id, n_ratings),
    'rating': np.random.randint(1, 6, n_ratings)  # Ratings from 1 to 5
}
ratings_df = pd.DataFrame(ratings_data)

# Remove duplicates and create interaction matrix
ratings_df = ratings_df.drop_duplicates(['user_id', 'movie_id'])

print("Dataset shapes:")
print(f"Movies: {movies_df.shape}")
print(f"Users: {users_df.shape}")
print(f"Ratings: {ratings_df.shape}")

# Preview the data
print("\nMovies sample:")
print(movies_df.head(3))
print("\nRatings sample:")
print(ratings_df.head(3))

## Content-Based Filtering

First, we'll implement the content-based filtering component using TF-IDF vectorization on movie descriptions and genres.

In [ ]:
class ContentBasedRecommender:
    def __init__(self):
        self.tfidf = TfidfVectorizer(stop_words='english')
        self.item_features = None
        self.item_ids = None
        self.similarity_matrix = None
        
    def fit(self, movies_df):
        # Combine genre and description for feature extraction
        text_features = movies_df['genre'] + ' ' + movies_df['description']
        
        # Create TF-IDF matrix
        self.item_features = self.tfidf.fit_transform(text_features)
        
        # Compute similarity matrix
        self.similarity_matrix = cosine_similarity(self.item_features)
        
        # Store movie IDs for later use
        self.item_ids = movies_df['movie_id'].values
        
    def recommend(self, user_id, ratings_df, n_recommendations=5):
        # Get user's rated movies
        user_ratings = ratings_df[ratings_df['user_id'] == user_id]
        if len(user_ratings) == 0:
            return []
        
        # Calculate weighted average of similarities
        weighted_sims = np.zeros(len(self.item_ids))
        for _, row in user_ratings.iterrows():
            movie_idx = np.where(self.item_ids == row['movie_id'])[0][0]
            weighted_sims += self.similarity_matrix[movie_idx] * row['rating']
            
        weighted_sims /= len(user_ratings)
        
        # Get top N recommendations (excluding already rated items)
        rated_items = user_ratings['movie_id'].values
        mask = np.isin(self.item_ids, rated_items, invert=True)
        
        recommendations = []
        for idx in np.argsort(weighted_sims * mask)[::-1][:n_recommendations]:
            recommendations.append({
                'movie_id': self.item_ids[idx],
                'score': weighted_sims[idx]
            })
            
        return recommendations

# Create and train the content-based recommender
cb_recommender = ContentBasedRecommender()
cb_recommender.fit(movies_df)

# Test the recommender for a sample user
test_user_id = 0
cb_recommendations = cb_recommender.recommend(test_user_id, ratings_df)
print("Content-based recommendations for user 0:")
for rec in cb_recommendations:
    movie = movies_df[movies_df['movie_id'] == rec['movie_id']].iloc[0]
    print(f"Movie: {movie['title']}, Score: {rec['score']:.3f}")

## Collaborative Filtering

Now we'll implement collaborative filtering using matrix factorization (SVD).

In [ ]:
class CollaborativeFilteringRecommender:
    def __init__(self, n_factors=50):
        self.n_factors = n_factors
        self.svd = TruncatedSVD(n_components=n_factors)
        self.user_features = None
        self.item_features = None
        self.user_ids = None
        self.item_ids = None
        
    def fit(self, ratings_df):
        # Create user-item matrix
        user_item_matrix = pd.pivot_table(
            ratings_df,
            values='rating',
            index='user_id',
            columns='movie_id',
            fill_value=0
        )
        
        # Store IDs
        self.user_ids = user_item_matrix.index.values
        self.item_ids = user_item_matrix.columns.values
        
        # Perform matrix factorization
        self.item_features = self.svd.fit_transform(user_item_matrix)
        self.user_features = user_item_matrix.dot(self.item_features.dot(np.linalg.pinv(np.diag(self.svd.singular_values_))))
        
    def recommend(self, user_id, ratings_df, n_recommendations=5):
        if user_id not in self.user_ids:
            return []
            
        # Get user's feature vector
        user_idx = np.where(self.user_ids == user_id)[0][0]
        user_vector = self.user_features[user_idx]
        
        # Calculate predicted ratings
        predicted_ratings = user_vector.dot(self.svd.components_)
        
        # Get user's rated items
        rated_items = ratings_df[ratings_df['user_id'] == user_id]['movie_id'].values
        
        # Create mask for unrated items
        mask = np.isin(self.item_ids, rated_items, invert=True)
        
        # Get top N recommendations
        recommendations = []
        for idx in np.argsort(predicted_ratings * mask)[::-1][:n_recommendations]:
            recommendations.append({
                'movie_id': self.item_ids[idx],
                'score': predicted_ratings[idx]
            })
            
        return recommendations

# Create and train the collaborative filtering recommender
cf_recommender = CollaborativeFilteringRecommender()
cf_recommender.fit(ratings_df)

# Test the recommender for the same user
cf_recommendations = cf_recommender.recommend(test_user_id, ratings_df)
print("Collaborative filtering recommendations for user 0:")
for rec in cf_recommendations:
    movie = movies_df[movies_df['movie_id'] == rec['movie_id']].iloc[0]
    print(f"Movie: {movie['title']}, Score: {rec['score']:.3f}")

## Hybrid Recommender

Now we'll implement the hybrid recommender that combines both approaches using a weighted average of their scores.

In [ ]:
class HybridRecommender:
    def __init__(self, cb_recommender, cf_recommender, alpha=0.5):
        self.cb_recommender = cb_recommender
        self.cf_recommender = cf_recommender
        self.alpha = alpha
        
    def recommend(self, user_id, ratings_df, n_recommendations=5):
        # Get recommendations from both systems
        cb_recs = self.cb_recommender.recommend(user_id, ratings_df, n_recommendations=n_recommendations*2)
        cf_recs = self.cf_recommender.recommend(user_id, ratings_df, n_recommendations=n_recommendations*2)
        
        # Create dictionaries for easy lookup
        cb_scores = {rec['movie_id']: rec['score'] for rec in cb_recs}
        cf_scores = {rec['movie_id']: rec['score'] for rec in cf_recs}
        
        # Combine and normalize scores
        all_movies = set(cb_scores.keys()) | set(cf_scores.keys())
        hybrid_scores = {}
        
        for movie_id in all_movies:
            cb_score = cb_scores.get(movie_id, 0)
            cf_score = cf_scores.get(movie_id, 0)
            
            # Combine scores using weighted average
            hybrid_scores[movie_id] = (
                self.alpha * cf_score + 
                (1 - self.alpha) * cb_score
            )
        
        # Sort and get top N recommendations
        sorted_items = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)
        
        recommendations = []
        for movie_id, score in sorted_items[:n_recommendations]:
            recommendations.append({
                'movie_id': movie_id,
                'score': score
            })
            
        return recommendations

# Create and test hybrid recommender
hybrid_recommender = HybridRecommender(cb_recommender, cf_recommender, alpha=0.7)
hybrid_recommendations = hybrid_recommender.recommend(test_user_id, ratings_df)

print("Hybrid recommendations for user 0:")
for rec in hybrid_recommendations:
    movie = movies_df[movies_df['movie_id'] == rec['movie_id']].iloc[0]
    print(f"Movie: {movie['title']}, Score: {rec['score']:.3f}")

## Evaluation

Let's implement some common evaluation metrics for recommender systems.

In [ ]:
def precision_at_k(recommended_items, relevant_items, k):
    """Calculate precision@k for a single user"""
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    
    hits = len(set(recommended_items) & set(relevant_items))
    return hits / k if k > 0 else 0

def recall_at_k(recommended_items, relevant_items, k):
    """Calculate recall@k for a single user"""
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
        
    hits = len(set(recommended_items) & set(relevant_items))
    return hits / len(relevant_items) if len(relevant_items) > 0 else 0

def evaluate_recommender(recommender, test_users, ratings_df, k=5):
    """Evaluate a recommender system using precision@k and recall@k"""
    precisions = []
    recalls = []
    
    # Split data into train/test
    train_data = ratings_df.sample(frac=0.8, random_state=42)
    test_data = ratings_df.drop(train_data.index)
    
    for user_id in test_users:
        # Get recommendations
        recs = recommender.recommend(user_id, train_data, n_recommendations=k)
        recommended_items = [rec['movie_id'] for rec in recs]
        
        # Get relevant items from test set
        relevant_items = test_data[
            (test_data['user_id'] == user_id) & 
            (test_data['rating'] >= 4)  # Consider ratings >= 4 as relevant
        ]['movie_id'].values
        
        # Calculate metrics
        prec = precision_at_k(recommended_items, relevant_items, k)
        rec = recall_at_k(recommended_items, relevant_items, k)
        
        precisions.append(prec)
        recalls.append(rec)
    
    return {
        'precision@k': np.mean(precisions),
        'recall@k': np.mean(recalls)
    }

# Evaluate all recommenders
test_users = users_df['user_id'].values[:10]  # Test with first 10 users
k = 5

print("Evaluating Content-Based Recommender:")
cb_metrics = evaluate_recommender(cb_recommender, test_users, ratings_df, k)
print(f"Precision@{k}: {cb_metrics['precision@k']:.3f}")
print(f"Recall@{k}: {cb_metrics['recall@k']:.3f}")

print("\nEvaluating Collaborative Filtering Recommender:")
cf_metrics = evaluate_recommender(cf_recommender, test_users, ratings_df, k)
print(f"Precision@{k}: {cf_metrics['precision@k']:.3f}")
print(f"Recall@{k}: {cf_metrics['recall@k']:.3f}")

print("\nEvaluating Hybrid Recommender:")
hybrid_metrics = evaluate_recommender(hybrid_recommender, test_users, ratings_df, k)
print(f"Precision@{k}: {hybrid_metrics['precision@k']:.3f}")
print(f"Recall@{k}: {hybrid_metrics['recall@k']:.3f}")

## Grid Search for Optimal Alpha

Let's find the best alpha value for our hybrid recommender by performing a grid search.

In [ ]:
# Perform grid search over alpha values
alpha_values = np.linspace(0, 1, 11)  # 0 to 1 in steps of 0.1
results = []

for alpha in alpha_values:
    hybrid_recommender.alpha = alpha
    metrics = evaluate_recommender(hybrid_recommender, test_users, ratings_df, k)
    results.append({
        'alpha': alpha,
        'precision': metrics['precision@k'],
        'recall': metrics['recall@k']
    })

# Convert results to DataFrame for plotting
results_df = pd.DataFrame(results)

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(results_df['alpha'], results_df['precision'], label='Precision@k')
plt.plot(results_df['alpha'], results_df['recall'], label='Recall@k')
plt.xlabel('Alpha (weight of collaborative filtering)')
plt.ylabel('Metric Value')
plt.title('Performance Metrics vs Alpha')
plt.legend()
plt.grid(True)
plt.show()

# Find best alpha
best_result = max(results, key=lambda x: x['precision'])
print(f"\nBest alpha value: {best_result['alpha']:.2f}")
print(f"Best precision@{k}: {best_result['precision']:.3f}")
print(f"Corresponding recall@{k}: {best_result['recall']:.3f}")

## Save Models

Finally, let's save our trained models for future use.

In [ ]:
# Create models directory if it doesn't exist
import os
models_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'models')
os.makedirs(models_dir, exist_ok=True)

# Save the recommenders
joblib.dump(cb_recommender, os.path.join(models_dir, 'content_based_recommender.joblib'))
joblib.dump(cf_recommender, os.path.join(models_dir, 'collaborative_filtering_recommender.joblib'))
joblib.dump(hybrid_recommender, os.path.join(models_dir, 'hybrid_recommender.joblib'))

print("Models saved successfully!")